# Practical 12: A Pipeline You Can Re-run

**Tasks in a defensible order, failures that stop the line, and a schedule**

SCSE3040 Machine Learning Operations &middot; Bennett University &middot; Session 2026-27

| | |
|---|---|
| **Lectures this follows** | L26 |
| **Course Outcome** | CO4 |
| **Lab duration** | 120 minutes |
| **Memory needed** | about 400 MB |
| **Extra software** | nothing beyond the course environment |
| **Marks** | 10 |

## Aim

1. Divide a job into small tasks and state which depends on which.
2. Derive a safe running order automatically.
3. Make a pipeline safe to re-run, and safe to fail part way through.
4. Express a schedule, and distinguish batch from stream.

## What this practical is about

The work so far has been run by hand, in an order held in the operator's head.
That arrangement fails in two ways: the order is not recorded anywhere, and a
failure in the middle leaves the system in a state nobody can describe.

A **pipeline** states the work as tasks and the order as dependencies. Four
properties follow, and each is examined in turn below.

- **Order is derived, not written.** The dependencies are declared and the
  order is computed from them, so adding a task does not require renumbering
  anything.
- **A failure stops the line.** Tasks after a failure are skipped rather than
  attempted, which prevents a model being trained on data that failed
  validation.
- **A run is repeatable.** Running the whole pipeline twice produces the same
  result, which is what makes recovery from a failure trivial: correct the
  cause and run again.
- **A schedule is a setting.** The pipeline does not decide when it runs.

Airflow is the tool L26 uses for this. The mechanism is built here from
ordinary functions, because the mechanism is what is examined and it is small
enough to read in full.

## Before you start

P04 and P05 should be complete. Functions and tests can be written. Familiarity
with a Python dictionary and a list is assumed.

Run the cells from top to bottom. A later cell normally depends on an object
created by an earlier one, so a `NameError` usually indicates a skipped
prerequisite.

If notebook state becomes confusing, restart the kernel, clear all outputs, and
run again from the first cell.

In [ ]:
# Step 0: check the workbench before we start.
# This cell only looks; it changes nothing. Run it and read the last line.

import sys
from pathlib import Path

print("Python  :", sys.version.split()[0])
print("Folder  :", Path.cwd().name)

_missing = []
for _name in ['numpy', 'pandas', 'sklearn']:
    try:
        __import__(_name)
    except ImportError:
        _missing.append(_name)

for _name in ['numpy', 'pandas', 'sklearn']:
    _mark = "missing" if _name in _missing else "ok"
    print(f"  {_name:<14} {_mark}")

if _missing:
    print()
    print("STOP. Some libraries are missing:", ", ".join(_missing))
    print("Ask your instructor to run the setup in labs/SETUP.md.")
else:
    print()
    print("All good. You can carry on to Step 1.")

## Step 0b: ensure the shared dataset exists

Every practical in this course uses the same 600 synthetic delivery
observations. The next cell confirms the file is present and reconstructs it
from the course seed if it is absent.

In [ ]:
# The delivery-time dataset every practical in this course uses.
# If the file is missing we build it again from the same seed, so every
# student in the room gets byte-for-byte the same 600 rows.

import csv
from pathlib import Path

import numpy as np

SEED = 42
N_ROWS = 600
from pathlib import Path

def find_data(name="delivery_times.csv"):
    """Locate the shared dataset from any working directory.

    Walks upward from the current folder looking for data/<name>, so the
    notebook works whether it is launched from the practical folder, the
    Practicals folder, or the repository root."""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in (folder / "data" / name,
                          folder / "Practicals" / "data" / name):
            if candidate.exists():
                return candidate.resolve()
    raise FileNotFoundError(
        f"{name} not found in any data/ folder at or above {here}")

DATA = find_data()
print("dataset:", DATA)


def make_delivery_csv(path=DATA):
    """Write the 600-row delivery dataset. Same formula as the lectures."""
    rng = np.random.default_rng(SEED)
    distance_km = np.round(rng.uniform(0.5, 12.0, N_ROWS), 2)
    prep_time_min = np.round(rng.uniform(5, 30, N_ROWS), 0)
    traffic_level = rng.integers(1, 4, N_ROWS)
    rain = rng.binomial(1, 0.25, N_ROWS)
    delivery_min = np.round(
        6.0
        + 3.1 * distance_km
        + 0.65 * prep_time_min
        + 4.2 * traffic_level
        + 5.5 * rain
        + rng.normal(0, 2.5, N_ROWS),
        1,
    )
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["distance_km", "prep_time_min", "traffic_level",
                    "rain", "delivery_min"])
        for i in range(N_ROWS):
            w.writerow([distance_km[i], int(prep_time_min[i]),
                        int(traffic_level[i]), int(rain[i]), delivery_min[i]])
    return path


if not DATA.exists():
    make_delivery_csv()
    print("dataset rebuilt ->", DATA)
else:
    print("dataset found   ->", DATA)

# Walkthrough

Read each step, then run its cell.

## Step 1: write the tasks as ordinary functions

A pipeline is not a distinct kind of code. It is ordinary functions, each
performing one job, together with a description of the order.

Each task receives a shared `context` dictionary and adds to it. That is the
same arrangement Airflow uses to pass values between tasks, which is why the
functions here transfer without modification.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

WORK = Path("work")
WORK.mkdir(exist_ok=True)
FEATURES = ["distance_km", "prep_time_min", "traffic_level", "rain"]


def extract(context):
    """Read yesterday's deliveries."""
    context["orders"] = pd.read_csv(DATA)
    return f"read {len(context['orders'])} orders"


def validate(context):
    """Refuse to continue on obviously broken data."""
    orders = context["orders"]
    problems = []
    if orders["distance_km"].min() <= 0:
        problems.append("a distance of zero or less")
    if orders["delivery_min"].isna().any():
        problems.append("missing delivery times")
    if not orders["traffic_level"].isin([1, 2, 3]).all():
        problems.append("a traffic level outside 1-3")
    if problems:
        raise ValueError("; ".join(problems))
    return f"{len(orders)} rows look sane"


def transform(context):
    """Build the model's inputs."""
    orders = context["orders"]
    context["X"] = orders[FEATURES]
    context["y"] = orders["delivery_min"]
    return f"{context['X'].shape[1]} features prepared"


def train(context):
    """Fit the model and remember its error."""
    X_tr, X_te, y_tr, y_te = train_test_split(
        context["X"], context["y"], test_size=0.2, random_state=42
    )
    model = LinearRegression().fit(X_tr, y_tr)
    context["model"] = model
    context["mae"] = mean_absolute_error(y_te, model.predict(X_te))
    return f"MAE {context['mae']:.2f} minutes"


def report(context):
    """Write the one file a human will actually look at."""
    path = WORK / "daily_report.txt"
    path.write_text(
        f"rows: {len(context['orders'])}\n"
        f"mae_minutes: {context['mae']:.3f}\n",
        encoding="utf-8",
    )
    return f"wrote {path.name}"


print("five tasks defined")

## Step 2: describe the order as a graph

The dependencies are now declared. Read the dictionary as a statement that each
task waits for the tasks listed against it.

Observe that `report` waits for `train`, and `train` waits for `transform`.
Nothing here states when anything runs. It states only what must precede
what.

In [ ]:
DAG = {
    "extract":   [],
    "validate":  ["extract"],
    "transform": ["validate"],
    "train":     ["transform"],
    "report":    ["train"],
}

TASKS = {
    "extract": extract,
    "validate": validate,
    "transform": transform,
    "train": train,
    "report": report,
}

for name, waits_for in DAG.items():
    arrow = " <- " + ", ".join(waits_for) if waits_for else ""
    print(f"  {name}{arrow}")

## Step 3: derive a safe order

A **topological sort**: repeatedly take any task whose dependencies are all
complete. If at some point no task is ready while tasks remain, the graph
contains a cycle, and that condition is reported rather than allowed to hang.

In [ ]:
def plan(dag):
    """Return an order in which every task's dependencies come first."""
    order, done = [], set()
    remaining = dict(dag)
    while remaining:
        ready = [name for name, deps in remaining.items()
                 if all(d in done for d in deps)]
        if not ready:
            raise ValueError(
                "these tasks depend on each other in a loop: "
                + ", ".join(sorted(remaining))
            )
        for name in sorted(ready):        # sorted = same order always
            order.append(name)
            done.add(name)
            del remaining[name]
    return order


ORDER = plan(DAG)
print("run order:", " -> ".join(ORDER))

Sorting the ready list matters more than it appears. Without it the order could
differ between runs, and a pipeline that runs in a different order each night
cannot be debugged, because the symptom is not reproducible.

## Step 4: confirm the cycle check works

The word acyclic in DAG is a requirement rather than a description. The cell
below breaks it deliberately, because a check never observed to fire is a check
of unknown value.

In [ ]:
broken = {"a": ["b"], "b": ["c"], "c": ["a"]}

try:
    plan(broken)
except ValueError as problem:
    print("caught:", problem)

## Step 5: run the pipeline

The runner walks the derived order, calls each task, and records the outcome. If
a task raises, everything after it is **skipped**: not attempted, and not
silently passed over.

In [ ]:
import time


def run_dag(dag, tasks, context=None):
    """Run every task in a safe order. Stop the line on failure."""
    context = {} if context is None else context
    results, failed = [], False

    for name in plan(dag):
        if failed:
            print(f"  SKIPPED   {name}")
            results.append({"task": name, "status": "skipped"})
            continue

        started = time.perf_counter()
        try:
            message = tasks[name](context)
            took = (time.perf_counter() - started) * 1000
            print(f"  OK        {name:<10} {message}  "
                  f"({took:.0f} ms)")
            results.append({"task": name, "status": "ok",
                            "message": message})
        except Exception as problem:
            print(f"  FAILED    {name:<10} {type(problem).__name__}"
                  f": {problem}")
            results.append({"task": name, "status": "failed",
                            "message": str(problem)})
            failed = True

    print()
    print("PIPELINE", "FAILED" if failed else "SUCCEEDED")
    return results, context


results, context = run_dag(DAG, TASKS)

## Step 6: run it again and confirm nothing changed

This property is **idempotence**. Running the whole pipeline a second time and
comparing the report file establishes it.

A pipeline that is safe to re-run makes recovery from a failure trivial:
correct the cause and run again, with no manual cleanup in between.

In [ ]:
first_report = (WORK / "daily_report.txt").read_text(encoding="utf-8")

results_again, _ = run_dag(DAG, TASKS)
second_report = (WORK / "daily_report.txt").read_text(encoding="utf-8")

print()
print("first run :", first_report.strip().replace("\n", " | "))
print("second run:", second_report.strip().replace("\n", " | "))
print("identical :", first_report == second_report)

The two reports are identical for two reasons, both of them deliberate
choices. `report` writes the file rather than appending to it, and `train` uses
a fixed seed. Either omission would produce a pipeline that cannot be safely
re-run, and neither would announce itself.

## Step 7: observe a failure stopping the line

The cell below supplies invalid data, a delivery with a negative distance, and
`validate` refuses it.

The significant part is what does not happen. There is no training and no
report. A model trained on that data would have produced a plausible score and
nothing would have indicated a fault.

In [ ]:
bad = pd.read_csv(DATA).copy()
bad.loc[0, "distance_km"] = -5.0
bad_path = WORK / "bad_orders.csv"
bad.to_csv(bad_path, index=False)


def extract_bad(context):
    """Same as extract, but reads the corrupted file."""
    context["orders"] = pd.read_csv(bad_path)
    return f"read {len(context['orders'])} orders"


bad_tasks = dict(TASKS, extract=extract_bad)
bad_results, _ = run_dag(DAG, bad_tasks)

Three tasks skipped. Yesterday's report remains on disk, unchanged and still
correct, which is a better outcome than a fresh report computed from invalid
data.

## Step 8: retry only what is worth retrying

Not every failure should stop the line. A transient network failure merits a
second attempt. Invalid data does not.

Airflow provides a `retries` setting on every task. The same idea is
implemented below at a size that can be read in full.

In [ ]:
attempts_made = {"count": 0}


def flaky_download(context):
    """Fails the first two times, like a real network."""
    attempts_made["count"] += 1
    if attempts_made["count"] < 3:
        raise ConnectionError("the weather service did not answer")
    return f"downloaded on attempt {attempts_made['count']}"


def with_retries(task, tries=3, wait=0.1):
    """Wrap a task so it is retried before being given up on."""
    def wrapped(context):
        for attempt in range(1, tries + 1):
            try:
                return task(context)
            except Exception as problem:
                print(f"      attempt {attempt} failed: {problem}")
                if attempt == tries:
                    raise
                time.sleep(wait)
    return wrapped


print(with_retries(flaky_download)({}))

Retry what might succeed on a second attempt. Never retry a task that failed
because the data is wrong: the result is the same failure, three times, more
slowly, and with the cause three times further from the top of the log.

## Step 9: state when it should run

Five fields: minute, hour, day of month, month, day of week. An asterisk means
every value.

The same syntax appears in Airflow, in Kubernetes CronJobs, and in system
schedulers, so it is worth learning once properly.

One consequence from L26 is worth recalling. A schedule carries a
`start_date`, and Airflow's `catchup` defaults to true, so enabling a weekly
DAG whose start date is well in the past queues one run per missed interval. The
measured case was a start date of 1 August 2025 enabled on 19 November 2026,
which queued 68 runs at once.

In [ ]:
schedules = [
    ("0 2 * * *",   "every day at 02:00"),
    ("*/15 * * * *", "every 15 minutes"),
    ("0 9 * * 1",   "every Monday at 09:00"),
    ("0 0 1 * *",   "the first day of every month, midnight"),
    ("30 6 * * 1-5", "06:30 on weekdays only"),
]

print(f"{'cron':<15} {'means'}")
print("-" * 55)
for expression, meaning in schedules:
    print(f"{expression:<15} {meaning}")

print()
print("Our pipeline retrains overnight, so:  0 2 * * *")
print("Why 2 a.m.? Few orders, spare machines, and a human is")
print("awake by the time anyone needs the report.")

## Step 10: batch or stream

The final distinction from L26. Both are pipelines. They differ in what causes
a run to begin.

In [ ]:
print(f"{'':<18}{'BATCH':<28}{'STREAM'}")
print("-" * 74)
rows = [
    ("runs", "on a schedule", "continuously"),
    ("handles", "all of yesterday at once", "one event at a time"),
    ("delay", "hours", "seconds"),
    ("tools", "Airflow, cron", "Kafka, Flink"),
    ("good for", "retraining, reports", "fraud alerts, live ETA"),
    ("harder", "nothing much", "ordering, failures, cost"),
]
for label, batch, stream in rows:
    print(f"{label:<18}{batch:<28}{stream}")

print()
print("Our delivery service uses both: retrain nightly in batch,")
print("and serve predictions one request at a time in P07.")
print()
print("Choose batch unless you can say what breaks if the answer")
print("is an hour late. Streaming costs far more to run and to fix.")

# Your turn

The walkthrough is complete. The remaining work is yours.

There are **three tasks**, in increasing order of difficulty, each with a hint.

Where a `# TODO` comment appears, replace that line with your own code. Do not
rename the variable on the left of the `=` sign. The self-check at the end
looks for exactly those names.

When all three have been attempted, run the **self-check** cell. It may be run
as often as required and it never alters the notebook.

## Task T1: add a task to the pipeline

A copy of the trained model is to be saved every night, after training and
before the report.

1. Write a task function `save_model(context)` that writes `context["model"]`
   to `work/model.joblib` with `joblib.dump` and returns the string
   `"model saved"`.
2. Add it to a new graph `T1_dag` so that `save_model` waits for `train`, and
   `report` waits for `save_model`.
3. Run the pipeline with `run_dag` and keep the outcome in `T1_results`.

> **Hint.** Copy `DAG` with `dict(DAG)`, then set
> `T1_dag["save_model"] = ["train"]` and change `T1_dag["report"]` to
> `["save_model"]`. The order is derived from the graph, so nothing else needs
> renumbering.

In [ ]:
import joblib


def save_model(context):
    # TODO: dump context["model"] to work/model.joblib, return "model saved"
    return None


# TODO: a new graph with save_model between train and report
T1_dag = None

# TODO: run it
T1_results = None

print(T1_results)

## Task T2: predict the consequence of a failure

There is no code to run for this task. It is answered by reading the graph.

Using the graph from Step 2, where `extract` precedes `validate`, `transform`,
`train` and `report`, suppose `transform` fails.

Place the names of the tasks that would be skipped, in run order, in the list
`T2_skipped`. Then place the number of tasks that would report `ok` in
`T2_ok_count`.

> **Hint.** Everything before the failure has already succeeded. The failing
> task itself is recorded as failed and not as skipped. Everything after it is
> skipped.

In [ ]:
# TODO: which tasks are skipped when transform fails?
T2_skipped = None

# TODO: how many tasks report ok?
T2_ok_count = None

print("skipped:", T2_skipped, " ok:", T2_ok_count)

## Task T3: read and write a schedule

Two parts.

1. `T3_meaning`: state in your own words what `15 3 * * 0` means. The answer
   must be a string containing the hour `3`, the minute `15`, and the word
   `Sunday`.
2. `T3_cron`: write the cron expression for every day at 23:30.

> **Hint.** The five fields are minute, hour, day of month, month, day of week.
> In the day of week field, 0 is Sunday. The second part requires two numbers
> and three asterisks.

In [ ]:
# TODO: what does "15 3 * * 0" mean, in words?
T3_meaning = None

# TODO: cron for every day at 23:30
T3_cron = None

print(T3_meaning)
print(T3_cron)

## Self-check

Run the cell below to mark the work.

In [ ]:
# ------------------------------------------------------------------
# SELF-CHECK: run this when you have attempted the tasks above.
# It never breaks your notebook. A task you have not done yet simply
# shows FAIL.
# ------------------------------------------------------------------

_results = []


def _check(label, fn):
    """Evaluate one graded condition without ever raising."""
    try:
        ok = bool(fn())
    except Exception:
        ok = False
    _results.append((label, ok))


_check('T1 | save_model runs after train and before report', lambda: plan(T1_dag).index('train') < plan(T1_dag).index('save_model') < plan(T1_dag).index('report'))
_check('T1 | every task in the run succeeded', lambda: all(r['status'] == 'ok' for r in T1_results) and len(T1_results) == 6)
_check('T1 | the model file was really written', lambda: (WORK / 'model.joblib').is_file() and (WORK / 'model.joblib').stat().st_size > 100)
_check('T2 | train and report are skipped, in that order', lambda: list(T2_skipped) == ['train', 'report'])
_check('T2 | extract and validate are the two that succeeded', lambda: int(T2_ok_count) == 2)
_check('T3 | the explanation names the hour, the minute and the day', lambda: '3' in T3_meaning and '15' in T3_meaning and 'sunday' in T3_meaning.lower())
_check('T3 | T3_cron has the five cron fields', lambda: len(T3_cron.split()) == 5)
_check('T3 | it means 23:30 every day', lambda: T3_cron.split()[:2] == ['30', '23'] and T3_cron.split()[2:] == ['*', '*', '*'])

print("==================================================================")
print("SELF-CHECK   Practical 12: A Pipeline You Can Re-run")
print("==================================================================")
for _label, _ok in _results:
    print(f"  [{'PASS' if _ok else 'FAIL'}]  {_label}")
print("------------------------------------------------------------------")
_passed = sum(1 for _, _ok in _results if _ok)
print(f"  {_passed} of {len(_results)} checks passed")
print("==================================================================")
if _passed == len(_results):
    print("Well done. Save the notebook and submit it.")
else:
    print("Read the FAIL lines above, fix those tasks, run this cell again.")

## What to submit

1. This notebook, with every cell run and its output visible.
2. The file `work/daily_report.txt`.
3. In a markdown cell, one sentence stating what would have gone wrong had
   `validate` not stopped the pipeline in Step 7.

Name the file `P12_<your-roll-number>.ipynb` before uploading it.

A runnable version of this pipeline is provided as `work/pipeline.py`. Run it
with `python pipeline.py`, and with `python pipeline.py --fail train` to observe
the line stopping and the process exiting with code 1.

### How this practical is marked

| What is marked | Marks |
|---|---|
| Walkthrough run end to end, pipeline passing then failing safely | 3 |
| Task T1, a new task wired into the graph | 3 |
| Task T2, the effect of a failure predicted correctly | 2 |
| Task T3, schedules read and written | 2 |
| **Total** | **10** |

## Read more

- Apache Airflow, core concepts: <https://airflow.apache.org/docs/apache-airflow/stable/core-concepts/index.html>
- Apache Airflow, DAG scheduling and catchup: <https://airflow.apache.org/docs/apache-airflow/stable/authoring-and-scheduling/dag-run.html>
- Kubernetes, CronJob: <https://kubernetes.io/docs/concepts/workloads/controllers/cron-jobs/>
- crontab guru, for reading cron expressions: <https://crontab.guru/>